## **Problem Statement**

Dream Housing Finance wants to automate real-time loan approval decisions. They've collected applicant data (income, credit history,education, etc.) and want a model that predicts Loan_Status (Y/N) for new applicants in the test set.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('./data/train.csv')

## **Data Inspect**

Here, we talk a quick look at the data

In [3]:
# Taking a quick look on the data. So we can have an idea of what we are working with.

df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


**Load ID :** Unique ID for each loan application.

**Gender :** Male or Female.

**Married :** Married or Unmarried.

**Dependents :** Number of dependents.

**Education :** Education level.

**Self_Employed :** Self-employed or not.

**ApplicantIncome :** Applicant income per month.

**CoapplicantIncome :** Coapplicant income per month.

**LoanAmount :** Loan amount in thousands.

**Loan_Amount_Term :** Loan duration in months.

**Credit_History :** Credit history meets guidelines or not.

**Property_Area :** Urban, Semi-Urban or Rural.

**Loan_Status :** Loan approved or not.

**Note :** We don't know if the data is based on India or US (i.e.) Whether the currency is based on INR or USD. 

In [4]:
# Checking the shape of the data. Because, we should know how many rows and columns the data has.

df.shape

(614, 13)

The data has 614 rows and 13 columns.

In [5]:
# Checking the data types of the columns. Because, we should know what type of data each column has.

df.dtypes

Loan_ID                  str
Gender                   str
Married                  str
Dependents               str
Education                str
Self_Employed            str
ApplicantIncome        int64
CoapplicantIncome    float64
LoanAmount           float64
Loan_Amount_Term     float64
Credit_History       float64
Property_Area            str
Loan_Status              str
dtype: object

In [21]:
# Checking what values are present in each column. Because, we should if those values are valid and whether they along with the datatypes listed above.

for col in df.columns:
    if col not in ['Loan_ID', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term']:
        print(f"{col} : {list(df[col].unique())}")

Gender : ['Male', 'Female', nan]
Married : ['No', 'Yes', nan]
Dependents : ['0', '1', '2', '3+', nan]
Education : ['Graduate', 'Not Graduate']
Self_Employed : ['No', 'Yes', nan]
Credit_History : [np.float64(1.0), np.float64(0.0), np.float64(nan)]
Property_Area : ['Urban', 'Rural', 'Semiurban']
Loan_Status : ['Y', 'N']


Note : Here we only considered the columns which are categorical. Numerical columns are better summarized using .describe().

So, there are "nan" values for the following columns : LoanAmount, Loan_Amount_Term, Credit_History. Otherwise, no problem at the surface level. Except "Dependents"

**PROBLEM :** "Dependents column has a value "3+". If we convert it to just 3, then we are essentially saying that 4 or 5 = 3. Which is not true. Will deal with it later.

In [7]:
# Taking a quick look on numerical column. So that we can ensure if they have any problems even at the surface level.

df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


**Applicant Income :** 

- The data is right skewed. Because, mean >> median. 
- The person with max income is 14x greater than person at 75th percentile.

**Coapplicant Income :** 

- The first 25% is 0 means 25% of our applicants have no co-applicant income. (May be most of them are not married or something like that)
- Same right skewed. Because, mean >> median and here the person with max income is 18x greater than person at 75th percentile.

**Note :** Both Applicant Income and Coapplicant Income are right skewed and they seem to have outliers. We might want to use transformation later.

**Loan Amount & Loan Amount Term:** 

- It seems like the data has some null values as the count of rows != count of values in LoanAmount column.
- **Loan Amount** has *22 null values*.
- **Loan Amount Term** has *14 null values*.
- **Loan Amount Term** has 360 (i.e.) *30 years* as a more common number. So, there is not a lot this column can help us with.


In [19]:
# Checking the distribution of the target variable. So, we could determine if the target variable is imbalanced or not.

target_cnt = df['Loan_Status'].value_counts().to_list()
approved_percentage = round((target_cnt[0] / sum(target_cnt))*100, 2)
not_approved_percentage = round((target_cnt[1] / sum(target_cnt))*100, 2)

print(f"Approved Percentage : {approved_percentage}%\nNot Approved Percentage : {not_approved_percentage}%")

Approved Percentage : 68.73%
Not Approved Percentage : 31.27%


We have 68.73% approved loans and 31.27% not approved loans. So, the dataset is imbalanced. 

We need to be careful. Because our model can still approve all loans and get 68.73% accuracy.

# **Data Audit**